# RAG Poulina – Chatbot IA pour les souches d'élevage
### Pipeline : CSV → Embeddings → Qdrant → Mistral

Ce notebook implémente un système **RAG (Retrieval-Augmented Generation)** complet :
1. **Ingestion** : chargement et nettoyage du CSV d'analyses
2. **Chunking** : découpage des données en chunks sémantiques
3. **Embeddings** : vectorisation avec `sentence-transformers` (local, gratuit)
4. **Vector Store** : stockage dans **Qdrant** (in-memory pour le dev)
5. **Retrieval** : recherche des chunks les plus pertinents
6. **Generation** : réponse finale avec **Mistral**


In [21]:
%pip install -r requirements.txt -qU
%pip install langchain_mistralai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [22]:
from dotenv import load_dotenv
import os
import sys

dotenv_path = "C:\\Users\\lysia\\Documents\\IUT1\\Stage\\Projet\\ChatbotIA\\.env"
load_dotenv(dotenv_path=dotenv_path)

from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

# LLM utilisé pour tous les exercices
llm = ChatMistralAI(model="mistral-small", temperature=0.0)
print("LLM initialisé.")


LLM initialisé.


## 2. Imports & Configuration

In [23]:
import os
import json
import uuid
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

import anthropic
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)
from sentence_transformers import SentenceTransformer

# ── Modèles ────────────────────────────────────────────────────────────────
from langchain_mistralai import MistralAIEmbeddings
EMBED_MODEL   = "intfloat/multilingual-e5-base"   # multilingue, fonctionne en FR/AR
MISTRAL_MODEL  = "mistral-large-latest"


# ── Qdrant (in-memory pour dev, remplacer par URL pour prod) ───────────────
COLLECTION    = "poulina_analyses"
VECTOR_DIM    = 768  # dimension de multilingual-e5-base

# ── Paramètres RAG ─────────────────────────────────────────────────────────
TOP_K         = 5    # nombre de chunks récupérés par requête
SCORE_THRESH  = 0.35 # seuil de similarité minimum

print("✅ Imports OK")
print(f"   Embed model  : {EMBED_MODEL}")
print(f"   Mistral model : {MISTRAL_MODEL}")

✅ Imports OK
   Embed model  : intfloat/multilingual-e5-base
   Mistral model : mistral-large-latest


## 3. Chargement du CSV

In [24]:
CSV_PATH = "C:\\Users\\lysia\\Documents\\IUT1\\Stage\\Projet\\ChatbotIA\\poulina_dataset_5000.csv"  

if Path(CSV_PATH).exists():
    df = pd.read_csv(CSV_PATH)
    print(f"✅ Fichier chargé : {len(df)} lignes, {len(df.columns)} colonnes")
else : 
    print("erreur csv")

✅ Fichier chargé : 5000 lignes, 20 colonnes


## 4. Nettoyage & Normalisation

In [25]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Normalise les colonnes booléennes
    for col in ["effectuee", "conforme"]:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: True if str(x).lower() in ("true","1","oui","yes") else False
            )
    # Remplace les NaN textuels
    df = df.fillna("N/A")
    return df

df = clean_df(df)

# ── Statistiques rapides ───────────────────────────────────────────────────
print("Statistiques globales")
print(f"   Total analyses     : {len(df)}")
if "conforme" in df.columns:
    pct_nc = (~df["conforme"]).sum() / len(df) * 100
    print(f"   Non conformes      : {pct_nc:.1f}%")
if "meilleure_souche" in df.columns:
    print(f"   Souches distinctes : {df['meilleure_souche'].nunique()}")
    print(f"   Top souches:\n{df['meilleure_souche'].value_counts().head(5).to_string()}")


Statistiques globales
   Total analyses     : 5000
   Souches distinctes : 4
   Top souches:
meilleure_souche
Ross 308            1648
Lohmann Brown       1461
Cobb 500            1140
Hybrid Converter     751


## 5. Chunking


In [26]:
def row_to_chunk(row: pd.Series) -> str:
    """Convertit une ligne CSV en texte descriptif pour l'embedding."""
    parts = []

    if "id_centre" in row:
        parts.append(f"ID centre : {row['id_centre']}.")
    if "ville" in row:
        parts.append(f"Ville : {row['ville']}.")
    if "region" in row:
        parts.append(f"Région : {row['region']}.")
    if "type_production" in row:
        parts.append(f"Type de production : {row['type_production']}.")
    if "meilleure_souche" in row:
        parts.append(f"Meilleure souche : {row['meilleure_souche']}.")
    if "capacite" in row:
        parts.append(f"Capacité : {row['capacite']} unités.")
    if "temperature_moyenne" in row:
        parts.append(f"Température moyenne : {row['temperature_moyenne']} °C.")
    if "humidite" in row:
        parts.append(f"Humidité : {row['humidite']} %.")
    if "altitude" in row:
        parts.append(f"Altitude : {row['altitude']} m.")
    if "biosecurite_score" in row:
        parts.append(f"Score de biosécurité : {row['biosecurite_score']}.")
    if "historique_maladie" in row:
        parts.append(f"Historique maladie : {row['historique_maladie']}.")
    if "taux_mortalite" in row:
        parts.append(f"Taux de mortalité : {row['taux_mortalite']} %.")
    if "fertilite_visee" in row:
        parts.append(f"Fertilité visée : {row['fertilite_visee']} %.")
    if "cout_aliment" in row:
        parts.append(f"Coût aliment : {row['cout_aliment']}.")
    if "surface_m2" in row:
        parts.append(f"Surface : {row['surface_m2']} m².")
    if "experience_equipe" in row:
        parts.append(f"Expérience équipe : {row['experience_equipe']} ans.")
    if "distance_labo" in row:
        parts.append(f"Distance au labo : {row['distance_labo']} km.")
    if "demande_marche" in row:
        parts.append(f"Demande marché : {row['demande_marche']}.")
    if "saison" in row:
        parts.append(f"Saison : {row['saison']}.")
    if "budget" in row:
        parts.append(f"Budget : {row['budget']}.")

    return " ".join(parts)


# Génère tous les chunks avec leurs métadonnées
chunks = []
for _, row in df.iterrows():
    text = row_to_chunk(row)
    meta = {
        "id_centre"         : str(row.get("id_centre", "")),
        "ville"             : str(row.get("ville", "")),
        "region"            : str(row.get("region", "")),
        "type_production"   : str(row.get("type_production", "")),
        "meilleure_souche"  : str(row.get("meilleure_souche", "")),
        "capacite"          : str(row.get("capacite", "")),
        "biosecurite_score" : str(row.get("biosecurite_score", "")),
        "taux_mortalite"    : str(row.get("taux_mortalite", "")),
    }
    chunks.append({"id": str(uuid.uuid4()), "text": text, "metadata": meta})

print(f"✅ {len(chunks)} chunks générés")
print("\nExemple de chunk :")
print(chunks[0]["text"])


✅ 5000 chunks générés

Exemple de chunk :
ID centre : 1. Ville : Tunis. Région : Nord. Type de production : Oeuf. Meilleure souche : Lohmann Brown. Capacité : 6012 unités. Température moyenne : 25.5 °C. Humidité : 63 %. Altitude : 228 m. Score de biosécurité : 85. Historique maladie : Faible. Taux de mortalité : 6.7 %. Fertilité visée : 86 %. Coût aliment : 1.54. Surface : 5467 m². Expérience équipe : 2 ans. Distance au labo : 80 km. Demande marché : Moyenne. Saison : Hiver. Budget : Faible.


## 6. Génération des embeddings

Utilisation du modèle **multilingual-e5-base** : ~450 MB, fonctionne en FR/AR,  
tourne entièrement en local (pas de coût API).


In [28]:
print(f"⏳ Chargement du modèle {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✅ Modèle chargé")

def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Vectorise une liste de textes en batches."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
        batch = texts[i : i + batch_size]
        # Le préfixe 'query:' / 'passage:' améliore les scores e5
        vecs = embed_model.encode(
            ["passage: " + t for t in batch],
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

texts  = [c["text"] for c in chunks]
vectors = embed_texts(texts)

print(f"✅ Embeddings : shape {vectors.shape}")
VECTOR_DIM = vectors.shape[1]


⏳ Chargement du modèle intfloat/multilingual-e5-base ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7399.72it/s]


✅ Modèle chargé


Embeddings: 100%|██████████| 157/157 [08:04<00:00,  3.08s/it]

✅ Embeddings : shape (5000, 768)


## 7. Indexation dans Qdrant

**In-memory** pour le développement.  
Pour la production, remplace par :
```python
client = QdrantClient(url="http://localhost:6333")
# ou
client = QdrantClient(url="https://YOUR_CLOUD.qdrant.io", api_key="...")
```


In [29]:
# Client Qdrant in-memory
client = QdrantClient(url="http://localhost:6333")

# Création de la collection
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)

# Indexation par batches
BATCH = 100
points = [
    PointStruct(
        id=i,
        vector=vectors[i].tolist(),
        payload={"text": chunks[i]["text"], **chunks[i]["metadata"]},
    )
    for i in range(len(chunks))
]

for i in tqdm(range(0, len(points), BATCH), desc="Indexation Qdrant"):
    client.upsert(collection_name=COLLECTION, points=points[i:i+BATCH])

info = client.get_collection(COLLECTION)
print(f"✅ Collection '{COLLECTION}' : {info.points_count} points indexés")


C:\Users\lysia\AppData\Local\Temp\ipykernel_30344\91862241.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
c:\Users\lysia\AppData\Local\Programs\Python\Python314\Lib\site-packages\qdrant_client\qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


ResponseHandlingException: [WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée